Version: 02.14.2023

# Capstone Project: Bringing It All Together

In this lab, you will bring together many of the tools and techniques that you have learned throughout this course into a final project. You can choose from many different paths to get to the solution. You could use AWS Managed Services, such as Amazon Comprehend, or use the Amazon SageMaker models. Have fun on whichever path you choose.

### Business scenario

You work for a training organization that recently developed an introductory course about machine learning (ML). The course includes more than 40 videos that cover a broad range of ML topics. You have been asked to create an application that will students can use to quickly locate and view video content by searching for topics and key phrases.

You have downloaded all of the videos to an Amazon Simple Storage Service (Amazon S3) bucket. Your assignment is to produce a dashboard that meets your supervisor’s requirements.

To assist you, all of the previous labs have been provided in this workspace.

## Lab steps

To complete this lab, you will follow these steps:

1. [Viewing the video files](#1.-Viewing-the-video-files)
2. [Transcribing the videos](#2.-Transcribing-the-videos)
3. [Normalizing the text](#3.-Normalizing-the-text)
4. [Extracting key phrases and topics](#4.-Extracting-key-phrases-and-topics)
5. [Creating the dashboard](#5.-Creating-the-dashboard)

## Submitting your work

1. In the lab console, choose **Submit** to record your progress and when prompted, choose **Yes**.

1. If the results don't display after a couple of minutes, return to the top of these instructions and choose **Grades**.

     **Tip**: You can submit your work multiple times. After you change your work, choose **Submit** again. Your last submission is what will be recorded for this lab.

1. To find detailed feedback on your work, choose **Details** followed by **View Submission Report**.

## Useful information

The following cell contains some information that might be useful as you complete this project.

In [ ]:
bucket = "c217930a5501403l15995208t1w967515893790-labbucket-zec6jzc2fay6"
job_data_access_role = 'arn:aws:iam::967515893790:role/service-role/c217930a5501403l15995208t1-ComprehendDataAccessRole-CqJgxiMjujJL'

### Capstone setup

Run the package cell once. When it finishes, choose **Kernel → Restart Kernel**, then begin again from the existing bucket/role cell and continue in order. The versions below match the AWS Academy Python environment used by the preceding labs.


In [ ]:
%pip install -q "numpy==1.23.5" "scipy==1.11.4" "pandas==2.1.4" "scikit-learn==1.2.1" "ipywidgets==8.1.2"


In [ ]:
import os
import re
import json
import time
import html
import hashlib
import unicodedata
import subprocess
from pathlib import Path
from urllib.parse import quote

import boto3
import pandas as pd
import numpy as np

from botocore import UNSIGNED
from botocore.config import Config
from botocore.exceptions import ClientError

from IPython.display import display, HTML
import ipywidgets as widgets

# AWS clients
session = boto3.Session()
region = session.region_name or "us-east-1"

s3_client = session.client("s3", region_name=region)
public_s3_client = session.client(
    "s3",
    region_name=region,
    config=Config(signature_version=UNSIGNED),
)
transcribe_client = session.client("transcribe", region_name=region)
comprehend_client = session.client("comprehend", region_name=region)

# Source videos
SOURCE_BUCKET = "aws-tc-largeobjects"
SOURCE_PREFIX = "CUR-TF-200-ACMNLP-1/video/"

# Output locations in the lab bucket
CAPSTONE_PREFIX = "capstone"
TRANSCRIBE_OUTPUT_PREFIX = f"{CAPSTONE_PREFIX}/transcribe/json"
ARTIFACT_DIR = Path("capstone_artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

print("Region:", region)
print("Lab bucket:", bucket)
print("Comprehend role:", job_data_access_role)


## 1. Viewing the video files
([Go to top](#Capstone-8:-Bringing-It-All-Together))


The source video files are located in the following shared Amazon Simple Storage Service (Amazon S3) bucket.

In [ ]:
!aws s3 ls s3://aws-tc-largeobjects/CUR-TF-200-ACMNLP-1/video/

### 1A. Build the video catalog

Add and run this cell immediately after the existing `aws s3 ls` cell. It discovers every video programmatically and creates `videos_df`.


In [ ]:
VIDEO_EXTENSIONS = (".mp4", ".m4a", ".mp3", ".wav", ".flac", ".ogg", ".webm", ".mov")

def list_public_video_keys():
    """List all video objects. Fall back to the AWS CLI if unsigned boto3 listing is blocked."""
    keys = []
    try:
        paginator = public_s3_client.get_paginator("list_objects_v2")
        for page in paginator.paginate(Bucket=SOURCE_BUCKET, Prefix=SOURCE_PREFIX):
            for obj in page.get("Contents", []):
                key = obj["Key"]
                if key.lower().endswith(VIDEO_EXTENSIONS):
                    keys.append(key)
    except Exception as exc:
        print("Unsigned boto3 listing failed; using AWS CLI:", exc)
        cmd = [
            "aws", "s3", "ls",
            f"s3://{SOURCE_BUCKET}/{SOURCE_PREFIX}",
            "--recursive",
            "--no-sign-request",
        ]
        output = subprocess.check_output(cmd, text=True)
        for line in output.splitlines():
            parts = line.split(maxsplit=3)
            if len(parts) == 4:
                key = parts[3]
                if key.lower().endswith(VIDEO_EXTENSIONS):
                    keys.append(key)

    return sorted(set(keys))

video_keys = list_public_video_keys()

videos_df = pd.DataFrame({
    "video_id": range(1, len(video_keys) + 1),
    "video_key": video_keys,
})

videos_df["video_name"] = videos_df["video_key"].map(lambda x: Path(x).name)
videos_df["video_stem"] = videos_df["video_name"].map(lambda x: Path(x).stem)
videos_df["source_uri"] = videos_df["video_key"].map(
    lambda x: f"s3://{SOURCE_BUCKET}/{x}"
)
videos_df["video_url"] = videos_df["video_key"].map(
    lambda x: f"https://{SOURCE_BUCKET}.s3.amazonaws.com/{quote(x, safe='/')}"
)

if videos_df.empty:
    raise RuntimeError("No source videos were found. Re-run the original AWS CLI listing cell and verify access.")

print(f"Found {len(videos_df)} videos.")
display(videos_df[["video_id", "video_name", "source_uri"]])


## 2. Transcribing the videos
 ([Go to top](#Capstone-8:-Bringing-It-All-Together))

Use this section to implement your solution to transcribe the videos.

### 2A. Transcription helpers

The job names deliberately begin with `transcribe-job-`. AWS Academy lab policies can restrict allowed job-name prefixes, so do not rename that prefix.


In [ ]:
TERMINAL_TRANSCRIBE_STATES = {"COMPLETED", "FAILED"}

MEDIA_FORMATS = {
    ".mp4": "mp4",
    ".m4a": "mp4",
    ".mp3": "mp3",
    ".wav": "wav",
    ".flac": "flac",
    ".ogg": "ogg",
    ".webm": "webm",
    ".mov": "mp4",
}

def transcribe_identifiers(video_key):
    digest = hashlib.sha1(video_key.encode("utf-8")).hexdigest()[:12]
    job_name = f"transcribe-job-capstone-{digest}"
    output_key = f"{TRANSCRIBE_OUTPUT_PREFIX}/{digest}.json"
    return job_name, output_key

def get_transcribe_job(job_name):
    try:
        return transcribe_client.get_transcription_job(
            TranscriptionJobName=job_name
        )["TranscriptionJob"]
    except ClientError as exc:
        error_code = exc.response.get("Error", {}).get("Code", "")
        error_message = exc.response.get("Error", {}).get("Message", "").lower()
        if error_code in {"BadRequestException", "NotFoundException"} and (
            "not found" in error_message or "couldn't be found" in error_message
        ):
            return None
        raise

def start_or_reuse_transcribe_job(row):
    base_job_name, base_output_key = transcribe_identifiers(row.video_key)
    existing = get_transcribe_job(base_job_name)

    if existing is not None and existing["TranscriptionJobStatus"] != "FAILED":
        return {
            "video_id": row.video_id,
            "video_key": row.video_key,
            "video_name": row.video_name,
            "job_name": base_job_name,
            "output_key": base_output_key,
            "status": existing["TranscriptionJobStatus"],
            "failure_reason": existing.get("FailureReason", ""),
            "transcript_uri": existing.get("Transcript", {}).get("TranscriptFileUri", ""),
        }

    # If a deterministic job failed previously, create a fresh retry job while
    # retaining the AWS Academy-required 'transcribe-job-' prefix.
    if existing is not None and existing["TranscriptionJobStatus"] == "FAILED":
        retry_suffix = int(time.time())
        job_name = f"{base_job_name}-retry-{retry_suffix}"
        output_key = base_output_key.replace(".json", f"-retry-{retry_suffix}.json")
    else:
        job_name = base_job_name
        output_key = base_output_key

    extension = Path(row.video_name).suffix.lower()
    media_format = MEDIA_FORMATS.get(extension)
    if media_format is None:
        raise ValueError(f"Unsupported media format: {row.video_name}")

    # Retry transient service/concurrency throttling without creating duplicate jobs.
    for attempt in range(1, 7):
        try:
            transcribe_client.start_transcription_job(
                TranscriptionJobName=job_name,
                Media={"MediaFileUri": row.source_uri},
                MediaFormat=media_format,
                LanguageCode="en-US",
                OutputBucketName=bucket,
                OutputKey=output_key,
            )
            break
        except ClientError as exc:
            error_code = exc.response.get("Error", {}).get("Code", "")
            if error_code in {
                "LimitExceededException",
                "ThrottlingException",
                "TooManyRequestsException",
            } and attempt < 6:
                wait_seconds = min(60, 10 * attempt)
                print(f"Transcribe is busy; retrying in {wait_seconds} seconds...")
                time.sleep(wait_seconds)
                continue
            raise

    return {
        "video_id": row.video_id,
        "video_key": row.video_key,
        "video_name": row.video_name,
        "job_name": job_name,
        "output_key": output_key,
        "status": "QUEUED",
        "failure_reason": "",
        "transcript_uri": "",
    }

def refresh_transcribe_record(record):
    job = transcribe_client.get_transcription_job(
        TranscriptionJobName=record["job_name"]
    )["TranscriptionJob"]
    record["status"] = job["TranscriptionJobStatus"]
    record["failure_reason"] = job.get("FailureReason", "")
    record["transcript_uri"] = job.get("Transcript", {}).get("TranscriptFileUri", "")
    return record


### 2B. Start and monitor every transcription

Run this cell once and leave it running. It submits five videos at a time to avoid lab concurrency throttling. Re-running it is safe because completed deterministic job names are reused.


In [ ]:
MAX_CONCURRENT_JOBS = 5
POLL_SECONDS = 20

all_transcribe_records = []

for batch_start in range(0, len(videos_df), MAX_CONCURRENT_JOBS):
    batch = videos_df.iloc[batch_start:batch_start + MAX_CONCURRENT_JOBS]
    batch_records = []

    print(
        f"\nStarting/reusing videos "
        f"{batch_start + 1}-{batch_start + len(batch)} of {len(videos_df)}"
    )

    for row in batch.itertuples(index=False):
        try:
            record = start_or_reuse_transcribe_job(row)
            batch_records.append(record)
            print(f"{row.video_name}: {record['status']}")
        except ClientError as exc:
            error = exc.response.get("Error", {})
            code = error.get("Code", "")
            message = error.get("Message", str(exc))
            if code == "AccessDeniedException":
                raise PermissionError(
                    "Amazon Transcribe denied the request. Keep the required "
                    "'transcribe-job-' prefix and make sure this capstone was "
                    "opened from the active AWS Academy lab session."
                ) from exc
            raise RuntimeError(f"{row.video_name}: {code} - {message}") from exc

    while True:
        counts = {}
        for record in batch_records:
            if record["status"] not in TERMINAL_TRANSCRIBE_STATES:
                refresh_transcribe_record(record)
            counts[record["status"]] = counts.get(record["status"], 0) + 1

        print("Batch status:", counts)

        if all(r["status"] in TERMINAL_TRANSCRIBE_STATES for r in batch_records):
            break

        time.sleep(POLL_SECONDS)

    all_transcribe_records.extend(batch_records)

transcribe_jobs_df = pd.DataFrame(all_transcribe_records).sort_values("video_id")
display(transcribe_jobs_df[["video_id", "video_name", "status", "failure_reason"]])

failed_jobs = transcribe_jobs_df[transcribe_jobs_df["status"] != "COMPLETED"]
if not failed_jobs.empty:
    raise RuntimeError(
        "One or more transcription jobs failed. Inspect the displayed "
        "failure_reason values before continuing."
    )

print(f"All {len(transcribe_jobs_df)} transcription jobs completed.")


### 2C. Load the transcript JSON files

This cell reads the completed JSON outputs from your lab bucket and creates `transcripts_df`.


In [ ]:
def read_transcript_json(output_key, transcript_uri=""):
    try:
        response = s3_client.get_object(Bucket=bucket, Key=output_key)
        payload = json.loads(response["Body"].read().decode("utf-8"))
    except ClientError as exc:
        # Fallback to the URI returned by Transcribe if the expected key differs.
        if exc.response.get("Error", {}).get("Code") not in {"NoSuchKey", "404"}:
            raise
        if not transcript_uri:
            raise
        import requests
        response = requests.get(transcript_uri, timeout=60)
        response.raise_for_status()
        payload = response.json()

    transcript = payload.get("results", {}).get("transcripts", [])
    return transcript[0].get("transcript", "").strip() if transcript else ""

transcript_rows = []

for record in transcribe_jobs_df.to_dict("records"):
    text = read_transcript_json(
        record["output_key"],
        record.get("transcript_uri", ""),
    )
    transcript_rows.append({
        "video_id": record["video_id"],
        "video_key": record["video_key"],
        "video_name": record["video_name"],
        "transcript": text,
        "transcription_job": record["job_name"],
        "transcription_status": record["status"],
    })

transcripts_df = videos_df.merge(
    pd.DataFrame(transcript_rows),
    on=["video_id", "video_key", "video_name"],
    how="left",
)

missing_transcripts = transcripts_df["transcript"].fillna("").str.strip().eq("")
if missing_transcripts.any():
    display(transcripts_df.loc[missing_transcripts, ["video_name", "transcription_job"]])
    raise RuntimeError("At least one completed job produced an empty transcript.")

transcripts_df.to_csv(ARTIFACT_DIR / "video_transcripts.csv", index=False)

print(f"Loaded {len(transcripts_df)} non-empty transcripts.")
display(
    transcripts_df[["video_id", "video_name", "transcript"]]
    .assign(transcript=lambda d: d["transcript"].str.slice(0, 180) + "...")
    .head(10)
)


## 3. Normalizing the text
([Go to top](#Capstone-8:-Bringing-It-All-Together))

Use this section to perform any text normalization steps that are necessary for your solution.

### 3A. Normalize the transcripts

This normalization is dependency-light: Unicode normalization, lowercasing, URL removal, punctuation removal, whitespace cleanup, and English stop-word removal.


In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

STOP_WORDS = set(ENGLISH_STOP_WORDS)

def normalize_text(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = text.lower()
    tokens = re.findall(r"[a-z0-9]+(?:'[a-z0-9]+)?", text)
    normalized_tokens = [
        token
        for token in tokens
        if token not in STOP_WORDS and len(token) > 1
    ]
    return " ".join(normalized_tokens)

analysis_df = transcripts_df.copy()
analysis_df["normalized_text"] = analysis_df["transcript"].map(normalize_text)
analysis_df["word_count"] = analysis_df["transcript"].str.split().str.len()
analysis_df["normalized_word_count"] = (
    analysis_df["normalized_text"].str.split().str.len()
)

if analysis_df["normalized_text"].str.strip().eq("").any():
    raise RuntimeError("Normalization created an empty document.")

analysis_df.to_csv(ARTIFACT_DIR / "normalized_transcripts.csv", index=False)

display(
    analysis_df[
        ["video_id", "video_name", "word_count", "normalized_word_count", "normalized_text"]
    ].head()
)


## 4. Extracting key phrases and topics
([Go to top](#Capstone-8:-Bringing-It-All-Together))

Use this section to extract the key phrases and topics from the videos.

### 4A. Extract key phrases

Amazon Comprehend is used when permitted. If the lab role blocks synchronous key-phrase detection, the cell automatically falls back to per-document TF–IDF n-grams instead of stopping the project.


In [ ]:
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer

def split_text_by_utf8_bytes(text, max_bytes=4500):
    """Split text into Comprehend-safe chunks without breaking UTF-8 characters."""
    chunks = []
    current_words = []
    current_size = 0

    for word in text.split():
        word_size = len(word.encode("utf-8")) + 1
        if current_words and current_size + word_size > max_bytes:
            chunks.append(" ".join(current_words))
            current_words = [word]
            current_size = word_size
        else:
            current_words.append(word)
            current_size += word_size

    if current_words:
        chunks.append(" ".join(current_words))

    return chunks

def comprehend_key_phrases(text, top_n=15):
    aggregated = defaultdict(lambda: {"score": 0.0, "count": 0, "text": ""})

    for chunk in split_text_by_utf8_bytes(text):
        if not chunk.strip():
            continue

        response = comprehend_client.detect_key_phrases(
            Text=chunk,
            LanguageCode="en",
        )

        for item in response.get("KeyPhrases", []):
            phrase = item.get("Text", "").strip()
            if len(phrase) < 2:
                continue
            key = phrase.lower()
            aggregated[key]["text"] = phrase
            aggregated[key]["score"] = max(
                aggregated[key]["score"],
                float(item.get("Score", 0.0)),
            )
            aggregated[key]["count"] += 1

    ranked = sorted(
        aggregated.values(),
        key=lambda item: (item["score"], item["count"]),
        reverse=True,
    )
    return [item["text"] for item in ranked[:top_n]]

def local_tfidf_key_phrases(frame, top_n=15):
    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 3),
        max_df=0.95,
        min_df=1,
        max_features=15000,
        sublinear_tf=True,
    )
    matrix = vectorizer.fit_transform(frame["normalized_text"])
    terms = np.asarray(vectorizer.get_feature_names_out())

    results = []
    for row_index in range(matrix.shape[0]):
        row = matrix.getrow(row_index)
        if row.nnz == 0:
            results.append([])
            continue
        top_positions = row.data.argsort()[::-1][:top_n]
        feature_indices = row.indices[top_positions]
        results.append(terms[feature_indices].tolist())

    return results

use_comprehend = True

# Permission/availability probe using a small real transcript.
probe_text = analysis_df.iloc[0]["transcript"][:4000]
try:
    comprehend_client.detect_key_phrases(
        Text=probe_text,
        LanguageCode="en",
    )
except ClientError as exc:
    use_comprehend = False
    print(
        "Amazon Comprehend key-phrase API is unavailable in this lab role; "
        "using local TF-IDF key phrases instead."
    )
    print(exc.response.get("Error", {}).get("Code", str(exc)))

if use_comprehend:
    extracted_phrases = []
    for index, row in analysis_df.iterrows():
        phrases = comprehend_key_phrases(row["transcript"], top_n=15)
        extracted_phrases.append(phrases)
        print(f"{index + 1}/{len(analysis_df)}: {row['video_name']}")
        time.sleep(0.1)
else:
    extracted_phrases = local_tfidf_key_phrases(analysis_df, top_n=15)

analysis_df["key_phrases"] = extracted_phrases
analysis_df["key_phrases_text"] = analysis_df["key_phrases"].map(
    lambda items: " | ".join(items)
)
analysis_df["key_phrase_method"] = (
    "Amazon Comprehend" if use_comprehend else "TF-IDF fallback"
)

if analysis_df["key_phrases"].map(len).eq(0).any():
    raise RuntimeError("At least one video has no extracted key phrases.")

display(
    analysis_df[
        ["video_name", "key_phrase_method", "key_phrases_text"]
    ].head(10)
)


### 4B. Discover topics with NMF

This creates corpus-level topics and assigns each video its strongest topic.


In [ ]:
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer

topic_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95,
    max_features=10000,
    sublinear_tf=True,
)

topic_matrix = topic_vectorizer.fit_transform(analysis_df["normalized_text"])

if topic_matrix.shape[1] < 2:
    raise RuntimeError("The transcript corpus does not contain enough features for topic modeling.")

n_topics = min(
    8,
    max(2, len(analysis_df) // 5),
    topic_matrix.shape[0],
    topic_matrix.shape[1],
)

nmf_model = NMF(
    n_components=n_topics,
    init="nndsvda",
    random_state=42,
    max_iter=600,
)

document_topic_weights = nmf_model.fit_transform(topic_matrix)
topic_terms = np.asarray(topic_vectorizer.get_feature_names_out())

topic_rows = []
for topic_index, component in enumerate(nmf_model.components_):
    top_indices = component.argsort()[::-1][:12]
    keywords = topic_terms[top_indices].tolist()
    topic_rows.append({
        "topic_id": topic_index + 1,
        "topic_label": f"Topic {topic_index + 1}: " + ", ".join(keywords[:4]),
        "topic_keywords": keywords,
    })

topics_df = pd.DataFrame(topic_rows)

primary_topic_zero_based = document_topic_weights.argmax(axis=1)
analysis_df["topic_id"] = primary_topic_zero_based + 1
analysis_df["topic_score"] = document_topic_weights.max(axis=1)

topic_label_map = topics_df.set_index("topic_id")["topic_label"].to_dict()
topic_keywords_map = topics_df.set_index("topic_id")["topic_keywords"].to_dict()

analysis_df["topic_label"] = analysis_df["topic_id"].map(topic_label_map)
analysis_df["topic_keywords"] = analysis_df["topic_id"].map(topic_keywords_map)
analysis_df["topic_keywords_text"] = analysis_df["topic_keywords"].map(
    lambda items: " | ".join(items)
)

display(
    topics_df.assign(
        topic_keywords=topics_df["topic_keywords"].map(lambda x: " | ".join(x))
    )
)

display(
    analysis_df[
        ["video_name", "topic_label", "topic_score", "key_phrases_text"]
    ].head(10)
)


### 4C. Save analysis artifacts


In [ ]:
export_df = analysis_df.copy()
export_df["key_phrases"] = export_df["key_phrases"].map(
    lambda values: " | ".join(values)
)
export_df["topic_keywords"] = export_df["topic_keywords"].map(
    lambda values: " | ".join(values)
)

export_df.to_csv(ARTIFACT_DIR / "capstone_video_search_index.csv", index=False)

topics_export = topics_df.copy()
topics_export["topic_keywords"] = topics_export["topic_keywords"].map(
    lambda values: " | ".join(values)
)
topics_export.to_csv(ARTIFACT_DIR / "capstone_topics.csv", index=False)

print("Saved:")
print(ARTIFACT_DIR / "capstone_video_search_index.csv")
print(ARTIFACT_DIR / "capstone_topics.csv")


## 5. Creating the dashboard
([Go to top](#Capstone-8:-Bringing-It-All-Together))

Use this section to create the dashboard for your solution.

### 5A. Interactive search dashboard

Run this cell, enter a topic or phrase, optionally choose a discovered topic, and open the matching video link.


In [ ]:
search_box = widgets.Text(
    value="",
    placeholder="Example: regression, neural network, clustering",
    description="Search:",
    layout=widgets.Layout(width="70%"),
)

topic_options = [("All topics", 0)] + [
    (row.topic_label, int(row.topic_id))
    for row in topics_df.itertuples(index=False)
]

topic_dropdown = widgets.Dropdown(
    options=topic_options,
    value=0,
    description="Topic:",
    layout=widgets.Layout(width="70%"),
)

result_limit = widgets.IntSlider(
    value=10,
    min=1,
    max=min(25, len(analysis_df)),
    step=1,
    description="Results:",
    continuous_update=False,
)

dashboard_output = widgets.Output()

def make_snippet(text, query="", width=320):
    text = str(text)
    if not query:
        return text[:width] + ("..." if len(text) > width else "")

    lowered = text.lower()
    position = lowered.find(query.lower())
    if position < 0:
        return text[:width] + ("..." if len(text) > width else "")

    start = max(0, position - width // 3)
    end = min(len(text), start + width)
    prefix = "..." if start > 0 else ""
    suffix = "..." if end < len(text) else ""
    return prefix + text[start:end] + suffix

def render_dashboard(*_):
    query = search_box.value.strip().lower()
    selected_topic = int(topic_dropdown.value)

    frame = analysis_df.copy()
    searchable = (
        frame["video_name"].fillna("") + " " +
        frame["transcript"].fillna("") + " " +
        frame["normalized_text"].fillna("") + " " +
        frame["key_phrases_text"].fillna("") + " " +
        frame["topic_label"].fillna("") + " " +
        frame["topic_keywords_text"].fillna("")
    ).str.lower()

    if query:
        query_terms = query.split()
        mask = pd.Series(True, index=frame.index)
        for term in query_terms:
            mask &= searchable.str.contains(re.escape(term), regex=True)
        frame = frame.loc[mask].copy()
        frame["search_score"] = searchable.loc[frame.index].map(
            lambda text: sum(text.count(term) for term in query_terms)
        )
    else:
        frame["search_score"] = 0

    if selected_topic:
        frame = frame.loc[frame["topic_id"] == selected_topic].copy()

    frame = frame.sort_values(
        ["search_score", "topic_score"],
        ascending=[False, False],
    ).head(result_limit.value)

    with dashboard_output:
        dashboard_output.clear_output(wait=True)

        if frame.empty:
            display(HTML("<b>No matching videos found.</b>"))
            return

        display(HTML(f"<h3>{len(frame)} matching video(s)</h3>"))

        for row in frame.itertuples(index=False):
            snippet = make_snippet(row.transcript, query)
            card = f"""
            <div style="
                border:1px solid #d5d9d9;
                border-radius:8px;
                padding:14px;
                margin:10px 0;
                background:#ffffff;">
              <h4 style="margin:0 0 8px 0;">
                {html.escape(str(row.video_name))}
              </h4>
              <p><b>Topic:</b> {html.escape(str(row.topic_label))}</p>
              <p><b>Key phrases:</b> {html.escape(str(row.key_phrases_text))}</p>
              <p><b>Transcript:</b> {html.escape(snippet)}</p>
              <a href="{html.escape(str(row.video_url))}" target="_blank">
                Open video
              </a>
            </div>
            """
            display(HTML(card))

search_box.observe(render_dashboard, names="value")
topic_dropdown.observe(render_dashboard, names="value")
result_limit.observe(render_dashboard, names="value")

display(
    widgets.VBox([
        widgets.HTML(value="<h2>Machine Learning Course Video Search</h2>"),
        search_box,
        topic_dropdown,
        result_limit,
        dashboard_output,
    ])
)

render_dashboard()


### 5B. Export a standalone HTML dashboard

This creates `capstone_artifacts/capstone_dashboard.html`, which can be opened from the Jupyter file browser.


In [ ]:
dashboard_records = (
    analysis_df[
        [
            "video_name",
            "video_url",
            "transcript",
            "key_phrases_text",
            "topic_id",
            "topic_label",
            "topic_keywords_text",
        ]
    ]
    .fillna("")
    .to_dict("records")
)

dashboard_json = json.dumps(dashboard_records, ensure_ascii=False, default=str).replace("</", "<\\/")

topic_options_html = "\n".join(
    f'<option value="{int(row.topic_id)}">{html.escape(row.topic_label)}</option>'
    for row in topics_df.itertuples(index=False)
)

dashboard_html = f"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Machine Learning Course Video Search</title>
<style>
body {{ font-family: Arial, sans-serif; margin: 24px; background: #f7f8fa; }}
.controls {{ display: grid; grid-template-columns: 2fr 1fr; gap: 12px; margin-bottom: 16px; }}
input, select {{ padding: 10px; font-size: 16px; }}
.card {{ background: white; border: 1px solid #d5d9d9; border-radius: 8px; padding: 14px; margin: 10px 0; }}
.card h3 {{ margin-top: 0; }}
.meta {{ color: #444; }}
a {{ color: #0066c0; font-weight: 600; }}
@media (max-width: 700px) {{ .controls {{ grid-template-columns: 1fr; }} }}
</style>
</head>
<body>
<h1>Machine Learning Course Video Search</h1>
<p>Search transcripts, topics, and extracted key phrases.</p>
<div class="controls">
  <input id="query" placeholder="Search for a topic or phrase">
  <select id="topic">
    <option value="0">All topics</option>
    {topic_options_html}
  </select>
</div>
<p id="count"></p>
<div id="results"></div>
<script>
const DATA = {dashboard_json};
const queryInput = document.getElementById("query");
const topicSelect = document.getElementById("topic");
const results = document.getElementById("results");
const count = document.getElementById("count");

function escapeHtml(value) {{
  return String(value)
    .replaceAll("&", "&amp;")
    .replaceAll("<", "&lt;")
    .replaceAll(">", "&gt;")
    .replaceAll('"', "&quot;")
    .replaceAll("'", "&#039;");
}}

function render() {{
  const terms = queryInput.value.trim().toLowerCase().split(/\\s+/).filter(Boolean);
  const topic = Number(topicSelect.value);

  const filtered = DATA.filter(item => {{
    const searchable = [
      item.video_name,
      item.transcript,
      item.key_phrases_text,
      item.topic_label,
      item.topic_keywords_text
    ].join(" ").toLowerCase();

    const textMatches = terms.every(term => searchable.includes(term));
    const topicMatches = topic === 0 || Number(item.topic_id) === topic;
    return textMatches && topicMatches;
  }});

  count.textContent = `${{filtered.length}} matching video(s)`;

  results.innerHTML = filtered.map(item => `
    <div class="card">
      <h3>${{escapeHtml(item.video_name)}}</h3>
      <p class="meta"><b>Topic:</b> ${{escapeHtml(item.topic_label)}}</p>
      <p class="meta"><b>Key phrases:</b> ${{escapeHtml(item.key_phrases_text)}}</p>
      <p>${{escapeHtml(item.transcript.slice(0, 420))}}${{item.transcript.length > 420 ? "..." : ""}}</p>
      <a href="${{escapeHtml(item.video_url)}}" target="_blank" rel="noopener">Open video</a>
    </div>
  `).join("");
}}

queryInput.addEventListener("input", render);
topicSelect.addEventListener("change", render);
render();
</script>
</body>
</html>
"""

dashboard_path = ARTIFACT_DIR / "capstone_dashboard.html"
dashboard_path.write_text(dashboard_html, encoding="utf-8")

print("Standalone dashboard created:")
print(dashboard_path)
display(HTML(f'<a href="{dashboard_path}" target="_blank">Open the standalone dashboard</a>'))


### 5C. Final quality gate

Do not submit until every check below is `PASS`.


In [ ]:
quality_checks = pd.DataFrame([
    {
        "check": "All expected course videos were discovered",
        "passed": len(videos_df) >= 40,
        "value": len(videos_df),
    },
    {
        "check": "Every video has a completed transcription",
        "passed": (
            len(transcribe_jobs_df) == len(videos_df)
            and transcribe_jobs_df["status"].eq("COMPLETED").all()
        ),
        "value": f"{transcribe_jobs_df['status'].eq('COMPLETED').sum()}/{len(videos_df)}",
    },
    {
        "check": "Every transcript is non-empty",
        "passed": analysis_df["transcript"].fillna("").str.strip().ne("").all(),
        "value": int(analysis_df["transcript"].fillna("").str.strip().ne("").sum()),
    },
    {
        "check": "Every video has key phrases",
        "passed": analysis_df["key_phrases"].map(len).gt(0).all(),
        "value": int(analysis_df["key_phrases"].map(len).gt(0).sum()),
    },
    {
        "check": "Every video has a topic",
        "passed": analysis_df["topic_id"].notna().all(),
        "value": int(analysis_df["topic_id"].notna().sum()),
    },
    {
        "check": "Search index CSV exists",
        "passed": (ARTIFACT_DIR / "capstone_video_search_index.csv").exists(),
        "value": str(ARTIFACT_DIR / "capstone_video_search_index.csv"),
    },
    {
        "check": "Standalone dashboard exists",
        "passed": (ARTIFACT_DIR / "capstone_dashboard.html").exists(),
        "value": str(ARTIFACT_DIR / "capstone_dashboard.html"),
    },
])

quality_checks["status"] = np.where(quality_checks["passed"], "PASS", "FAIL")
display(quality_checks[["status", "check", "value"]])

if not quality_checks["passed"].all():
    raise RuntimeError("The capstone quality gate failed. Do not submit yet.")

print("CAPSTONE QUALITY GATE: PASS")


# Congratulations!

You have completed this lab, and you can now end the lab by following the lab guide instructions.

*©2023 Amazon Web Services, Inc. or its affiliates. All rights reserved. This work may not be reproduced or redistributed, in whole or in part, without prior written permission from Amazon Web Services, Inc. Commercial copying, lending, or selling is prohibited. All trademarks are the property of their owners.*
